In [1]:
!pip -q install pymupdf pytesseract pillow numpy tqdm sentence-transformers faiss-cpu transformers accelerate gradio
!apt-get update -qq
!apt-get install -y tesseract-ocr

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 30 not upgraded.


In [2]:
import os, io, re, json, glob, textwrap
import fitz
import faiss
import numpy as np
import pytesseract
from PIL import Image
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gradio as gr

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# cd /content/drive/MyDrive/RAG_chatbot

In [5]:
PDF_DIR = "/content/drive/MyDrive/RAG_chatbot/data"
DB_DIR = "/content/drive/MyDrive/RAG_chatbot/db"
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(DB_DIR, exist_ok=True)

EMBED_MODEL = "BAAI/bge-small-en-v1.5"

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_pdf_pages(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc, start=1):
        native_text = page.get_text("text").strip()
        text = native_text
        ocr_used = False

        if not native_text:
            pix = page.get_pixmap(dpi=180)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            text = pytesseract.image_to_string(img)
            ocr_used = True

        text = clean_text(text)
        if text:
            pages.append({
                "filename": os.path.basename(pdf_path),
                "page": page_num,
                "text": text,
                "ocr_used": ocr_used
            })
    doc.close()
    return pages

def chunk_text(text, chunk_size=900, overlap=200):
    words = text.split()
    if len(words) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(words):
        end = min(len(words), start + chunk_size)
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap
    return chunks

def build_chunks_from_pdf(pdf_path):
    pages = extract_pdf_pages(pdf_path)
    chunks = []
    for p in pages:
        for i, ch in enumerate(chunk_text(p["text"])):
            chunks.append({
                "id": f'{p["filename"]}_p{p["page"]}_c{i}',
                "filename": p["filename"],
                "page": p["page"],
                "chunk_id": i,
                "text": ch,
                "ocr_used": p["ocr_used"]
            })
    return chunks

In [6]:
pdf_files = glob.glob(os.path.join(PDF_DIR, "*.pdf"))
print("PDFs found:", len(pdf_files))

all_chunks = []
for pdf in tqdm(pdf_files):
    all_chunks.extend(build_chunks_from_pdf(pdf))

print("Chunks:", len(all_chunks))

PDFs found: 7


100%|██████████| 7/7 [00:18<00:00,  2.61s/it]

Chunks: 275


In [7]:
embedder = SentenceTransformer(EMBED_MODEL)

texts = [c["text"] for c in all_chunks]
embeddings = embedder.encode(texts, normalize_embeddings=True, batch_size=32, show_progress_bar=True)
embeddings = np.array(embeddings, dtype=np.float32)

dim = embeddings.shape[1]
index = faiss.IndexHNSWFlat(dim, 32)
index.hnsw.efConstruction = 200
faiss.normalize_L2(embeddings)
index.add(embeddings)

faiss.write_index(index, os.path.join(DB_DIR, "index.faiss"))
with open(os.path.join(DB_DIR, "chunks.json"), "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print("Saved FAISS index and chunks.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Saved FAISS index and chunks.


In [8]:
GEN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    device_map="auto",
    torch_dtype="auto"
)

gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300, # Explicitly use max_new_tokens and set a reasonable value
    do_sample=False,
    temperature=0.1
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [9]:
def retrieve(query, top_k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    q_emb = np.array(q_emb, dtype=np.float32)
    scores, idxs = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue
        item = all_chunks[idx].copy()
        item["score"] = float(score)
        results.append(item)
    return results

def make_context(results):
    return "\n\n".join(
        [f"[Source: {r['filename']} | Page {r['page']}]\n{r['text']}" for r in results]
    )

def rag_answer(question):
    results = retrieve(question, top_k=5)
    context = make_context(results)

    prompt = f"""You are a helpful RAG assistant.
Answer only using the context.
Cite sources like [filename | page X].

Context:
{context}

Question: {question}

Answer:"""

    out = gen(prompt)[0]["generated_text"]
    answer = out.split("Answer:")[-1].strip()

    sources = "\n".join([f"- {r['filename']} | page {r['page']}" for r in results])
    return answer + "\n\nSources:\n" + sources, results

In [10]:
ans, hits = rag_answer("Give me the bird sound production system details")
print(ans)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Chapter 1 Introduction
This chapter provides a brief introduction to the research on the classification and analysis of bird vocalizations using deep learning and computational modeling. It outlines the research background, the motivation behind the study, the avian sound production system, as well as the objectives and major contributions of the proposed work.

Figure 1.1: Avian Sound Production System
The intricate structure of the avian vocal tract not only enables sound production but also supports a wide range of vocal behaviors. Birds utilize this anatomical complexity to produce diverse acoustic signals used for communication, territory marking, and mating. These vocal signals vary significantly across species and can be broadly classified into calls and songs. Bird songs are generally more complex vocalizations than bird calls, which are considered simple, short sounds. While calls are often made by stringing together a series of simple sounds, songs are structured hierarchical

In [11]:
def chat_fn(message, history):
    ans, _ = rag_answer(message)
    return ans

demo = gr.ChatInterface(
    fn=chat_fn,
    title="PDF RAG Chatbot",
    description="Ask questions about your uploaded PDFs."
)
demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://55eb735f35a21d81b3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
